# Regression Modeling & Fine-Tuning by Chiemela Joseph Nwosu

In [ ]:
from google.colab import files
from IPython.display import Image

In [ ]:
uploaded = files.upload()

In [ ]:
Image('Mastering ChatGPT and Google Colab for Machine Learning Front.jpg',
      width = 300)

# Chapter 9: House Price Dataset - Data Preparation & Training for Regression Models


## `Chapter 9 Overview`

This chapter focuses on preparing data and training machine learning models for regression tasks, where the target variable represents a continuous numerical value.

Using a housing price dataset as a real-world example, I explore techniques tailored to regression analysis, including exploratory data analysis, feature preparation, and model training. The goal is to ensure the data is structured appropriately and that the resulting regression models produce accurate, interpretable, and reliable predictions.


### `Chapter Objectives`

By the end of this chapter, I aim to:

- Distinguish regression problems from classification problems
- Prepare continuous data for regression modeling
- Explore relationships between numerical housing features
- Train regression models to predict housing prices
- Evaluate regression performance using error-based metrics

## `Business Context: House Price Prediction`

Predicting housing prices is a common real-world regression problem with applications in real estate, finance, and urban planning. Accurate price estimates help buyers assess affordability, sellers determine competitive listing prices, and lenders evaluate risk when issuing mortgages.

In this dataset, housing prices serve as the continuous target variable, while property characteristics such as size, location, and structural features act as predictors. By modeling the relationship between these features and sale price, regression techniques can be used to support data-driven pricing decisions and market analysis.

### Dataset Variables & Descriptions

This dataset contains property sale records and home characteristics used to predict price (a continuous target variable).

`id`: Unique identifier for each house sale record

`date`: Date the house was sold (timestamp format)

`price`: Sale price of the home (target variable)

`Home Size and Structure`

* bedrooms: Number of bedrooms

* bathrooms: Number of bathrooms (can include partial bathrooms)

* sqft_living: Interior living space (square feet)

* sqft_lot: Total lot size (square feet)

* floors: Number of floors/levels in the home

* sqft_above: Square footage above ground

* sqft_basement: Basement square footage

`Quality, Condition, and Amenities`

* waterfront: Whether the home has waterfront access (0 = no, 1 = yes)

* view: Quality of view rating (ordinal scale; higher usually indicates better views)

* condition: Overall condition rating (ordinal scale)

* grade: Overall construction/design grade (ordinal scale; higher = better quality)

`Age and Renovation`

* yr_built: Year the home was originally built

* yr_renovated: Year the home was renovated (0 typically indicates never renovated)

`Location`

* zipcode: Postal code area of the home

* lat: Latitude coordinate

* long: Longitude coordinate

`Neighborhood Context`

* sqft_living15: Average living area of nearby homes (square feet)

* sqft_lot15: Average lot size of nearby homes (square feet)

The **price** variable serves as the continuous target, while the remaining variables act as predictors in the regression model.



## Part 1: Dataset Loading / EDA and Initial Inspection

Like in previous chapters, I begin by loading the dataset into a pandas DataFrame and inspecting its structure. This step helps me understand the available features, data types, and overall layout of the data before performing any transformations or analysis.


In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
import pandas as pd
import io
df = pd.read_csv(io.BytesIO(uploaded['House_Prices.csv']))

In [ ]:
df.shape

In [ ]:
df.head(), df.tail()

`Code Cell below is reading the file and displaying the basic statistics.`

In [ ]:
import pandas as pd

# Read the CSV file into a DataFrame
df = pd.read_csv("House_Prices.csv")

# Display basic statistics for numerical columns
basic_stats = df.describe()

print("Basic Statistics for Numerical Columns:")
print(basic_stats)

`Code Cell below is checking for missing values, value counts, unique values, etc.`

In [ ]:
import pandas as pd

# Read the CSV file into a DataFrame
df = pd.read_csv("House_Prices.csv")

# Display basic statistics for numerical columns
basic_stats = df.describe()

# Check for missing values
missing_values = df.isnull().sum()

# Display value counts and unique values for each column
value_counts = {}
unique_values = {}
for column in df.columns:
    value_counts[column] = df[column].value_counts()
    unique_values[column] = df[column].unique()

print("Basic Statistics for Numerical Columns:")
print(basic_stats)
print("\nMissing Values:")
print(missing_values)
print("\nValue Counts:")
for column, counts in value_counts.items():
    print(f"\n{column}:")
    print(counts)
print("\nUnique Values:")
for column, values in unique_values.items():
    print(f"\n{column}:")
    print(values)

### Quantifiable Insights from Initial Data Inspection

Based on the summary statistics and value distributions, I observed several data-driven insights that affect how I approach regression modeling:

`Data completeness`

* There are 0 missing values across all 21 columns, which means I can focus more on modeling decisions rather than imputation or missing-data bias.

`Price distribution (target variable)`

* The average price is approximately 540,088 USD, while the median price is 450,000 USD, which suggests the target is right-skewed (high-priced homes pull the mean upward).

* The middle 50% of home prices fall between 321,950 USD (Q1) and 645,000 (Q3), giving an IQR of about 323,050 USD. This indicates substantial price variability even among typical homes.

* The maximum price is $7,700,000, confirming the presence of extreme high-end outliers that may influence regression models sensitive to large values.

`Home size and typical structure`

* The average living area is about 2,080 sqft, with a median of 1,910 sqft, again suggesting right-skew in home size.

* The middle 50% of living area ranges from 1,427 sqft (Q1) to 2,550 sqft (Q3), giving an IQR of 1,123 sqft.

Bedroom counts are concentrated around common layouts:

* 3 bedrooms: 9,824 homes (most frequent)

* 4 bedrooms: 6,882 homes

Extreme values exist (e.g., 33 bedrooms), which likely represent data anomalies or rare properties and may require outlier handling.

`Amenities and rarity signals`

* Waterfront homes are rare: 163 out of 21,613 (about 0.75%). Even though rare, waterfront status could strongly affect price and may act as a high-impact predictor.

* Most homes have no view rating: 19,489 out of 21,613 (about 90.17%) have view = 0, meaning only about 9.83% of homes have a nonzero view score.

* Basements are present in a sizable portion of homes: 8,487 out of 21,613 (about 39.27%) have a nonzero basement square footage.

* Renovations are uncommon: 914 out of 21,613 (about 4.23%) show a nonzero renovation year, meaning most homes are not renovated (or renovations are not recorded).

`Floors distribution`

Housing structure is dominated by 1–2 story homes:

* 1 floor: 10,680 homes (about 49.41%)

* 2 floors: 8,241 homes (about 38.13%)

* 1.5 floors: 1,910 homes (about 8.84%)

* 3+ floors are rare (well under 5% combined)

These findings suggest the dataset contains meaningful predictors with real-world pricing relevance (size, grade, location, and amenities), while also including skew and outliers that may benefit from transformations (e.g., log scaling of price) or robust regression approaches.

## Part 2: Data Type Inspection and Transformation

Before performing feature analysis and regression modeling, it is important to understand how the dataset’s attributes are represented internally. In this section, I examine the data types of each column to identify numerical, categorical, and date-based features.

I also convert the date column to a proper datetime format to ensure it can be handled correctly in downstream analysis. Inspecting and adjusting data types early helps prevent modeling issues and supports accurate preprocessing decisions later in the regression workflow.


`The code cell below displays the number of columns per data type in the dataframe`

In [ ]:
import pandas as pd

# Read the CSV file into a DataFrame
df = pd.read_csv("House_Prices.csv")

# Get the data types of each column
column_types = df.dtypes

# Count the number of columns per data type
columns_per_type = column_types.value_counts()

# Display the result
print("Number of Columns Per Data Type:")
print(columns_per_type)

df.head(1)

`Code cell below converts the "date" column into a datetime data type then prints the data types of all columns`

In [ ]:
import pandas as pd

# Read the CSV file into a DataFrame
df = pd.read_csv("House_Prices.csv")

# Convert the "date" column to datetime data type
df['date'] = pd.to_datetime(df['date'])

# Print the data types of all columns
print("Data Types of All Columns:")
print(df.dtypes)

## Part 3: Feature Engineering & Analysis for Regression Model


`Prompt provided by Textbook: "In the 'House_Prices.csv' dataset [are there] some columns that are not useful for the house price prediction? (Moscato 175)`

* After providing ChatGPT the prompt and reading the output, it is discovered that the columns `id`, `date`, `yr_renovated`, `zipcode`, and `lat and long` are  not useful for house price prediction due less significant impact on the target variable `price`.

* `Lat and long` variable may be useful since the dataset does not include other location-based features.

* The next step is to create a script to remove the redundant columns from the dataframe and save a modified version into a new CSV file.

In [ ]:
import pandas as pd

# Read the CSV file into a DataFrame
df = pd.read_csv("House_Prices.csv")

# Remove the specified columns
columns_to_remove = ["id", "date", "yr_renovated"]
df = df.drop(columns=columns_to_remove)

# Save the modified DataFrame to a new CSV file
df.to_csv("House_Prices_Modified.csv", index=False)

print("Modified DataFrame saved to House_Prices_Modified.csv")

### Feature Analysis: Correlation Matrix

`Prompt provided by Textbook: In the dataset [are there] columns with very similar values? (Moscato 177)`

In [ ]:
import pandas as pd

# Read the CSV file into a DataFrame
df = pd.read_csv("House_Prices_Modified.csv")

# Calculate the correlation matrix for numerical columns
correlation_matrix = df.corr().abs()

# Filter out columns with correlation greater than a certain threshold
threshold = 0.9
highly_correlated_cols = (correlation_matrix > threshold) & (correlation_matrix < 1.0)
correlated_cols = [col for col in highly_correlated_cols.columns if any(highly_correlated_cols[col])]

print("Columns with highly correlated values:")
print(correlated_cols)

`The output list is empty, meaning that there are no columns where the values in one predicts the values in another.`

* The next step is to create a script that identifies the outliers and displays the analysis for each column.

`Prompt provided by Textbook: "Create a script to identify columns with outliers. Show the Analysis for each column." (Moscato 177)`

* The script in the code cell below will import the pandas library, upload the dataset, then analyzes each column looking for outliers and calculating the mean, standard deviation, etc. and prints the results at the end.

In [ ]:
import pandas as pd

# Read the CSV file into a DataFrame
df = pd.read_csv("House_Prices_Modified.csv")

# Function to identify outliers using the IQR method
def identify_outliers_iqr(column):
    q1 = column.quantile(0.25)
    q3 = column.quantile(0.75)
    iqr = q3 - q1
    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr
    outliers = column[(column < lower_bound) | (column > upper_bound)]
    return outliers

# Analyze each column for outliers
outliers_info = {}
for column in df.columns:
    if df[column].dtype in ['int64', 'float64']:
        outliers = identify_outliers_iqr(df[column])
        outliers_info[column] = {
            'total_outliers': len(outliers),
            'percentage_outliers': len(outliers) / len(df) * 100,
            'min_value': df[column].min(),
            'max_value': df[column].max(),
            'mean': df[column].mean(),
            'median': df[column].median(),
            'std_dev': df[column].std(),
        }

# Display analysis for each column with outliers
print("Analysis for Columns with Outliers:")
for column, info in outliers_info.items():
    if info['total_outliers'] > 0:
        print(f"Column: {column}")
        print(f"Total Outliers: {info['total_outliers']}")
        print(f"Percentage of Outliers: {info['percentage_outliers']:.2f}%")
        print(f"Minimum Value: {info['min_value']}")
        print(f"Maximum Value: {info['max_value']}")
        print(f"Mean: {info['mean']:.2f}")
        print(f"Median: {info['median']}")
        print(f"Standard Deviation: {info['std_dev']:.2f}")
        print()

To better understand the relationships between key housing attributes, I examine correlations between selected feature pairs that represent different aspects of residential properties. These pairings help identify which variables are most strongly associated with house prices and provide intuition for feature selection in regression modeling.

* The first pair analyzes `price` and `square footage of living space` (sqft_living). This relationship captures how home size influences market value and is typically one of the strongest predictors in housing datasets.

* The second pair focuses on `price` and `grade`. While square footage measures size, grade reflects overall construction quality, materials, and design. Examining this relationship helps explain price variation that cannot be attributed to size alone.

* The third pair evaluates `square footage of living space` and `number of bedrooms`. This comparison highlights how interior layout relates to total living area and can reveal diminishing returns when additional bedrooms are added without proportional increases in square footage.

Together, these correlations provide insight into how size, quality, and layout interact to influence housing prices and help inform subsequent regression modeling decisions.

`The next step is creating a scatter plot considering the three pairs.`


### `Feature Correlation Analysis: Scatter Plots`

The following scatter plots visualize the relationships between selected feature pairs identified in the correlation analysis. These plots help assess the direction, strength, and potential non-linear patterns between variables, as well as the presence of outliers. Understanding these relationships provides intuition for how each feature may contribute to the regression model.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Read the CSV file into a DataFrame
df = pd.read_csv("House_Prices.csv")

# Set up the figure and axes
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Scatter plot: Price vs. Square Footage of Living Area
sns.scatterplot(x='sqft_living', y='price', data=df, ax=axes[0])
axes[0].set_title('Price vs. Square Footage of Living Area')
axes[0].set_xlabel('Square Footage of Living Area')
axes[0].set_ylabel('Price')

# Scatter plot: Price vs. Grade
sns.scatterplot(x='grade', y='price', data=df, ax=axes[1])
axes[1].set_title('Price vs. Grade')
axes[1].set_xlabel('Grade')
axes[1].set_ylabel('Price')

# Scatter plot: Square Footage of Living Area vs. Number of Bedrooms
sns.scatterplot(x='sqft_living', y='bedrooms', data=df, ax=axes[2])
axes[2].set_title('Square Footage of Living Area vs. Number of Bedrooms')
axes[2].set_xlabel('Square Footage of Living Area')
axes[2].set_ylabel('Number of Bedrooms')

# Adjust layout
plt.tight_layout()

# Show the plot
plt.show()

### `Comprehensive Feature–Target Scatter Analysis`

To further explore how individual attributes relate to the target variable, I visualize scatter plots between price and all remaining features in the dataset. This comprehensive view helps identify linear and non-linear relationships, potential outliers, and features that may require transformation or scaling before regression modeling. Observing these patterns supports informed feature selection and model design decisions.


In [ ]:
import pandas as pd
import seaborn as sns

# Read the CSV file into a DataFrame
df = pd.read_csv("House_Prices.csv")

# Create a pairplot relating all attributes to the target variable (price)
sns.pairplot(
    df,
    x_vars=df.columns.drop(['id', 'date', 'price']),
    y_vars='price',
    diag_kind=None,
    height=4
)

# Show the plot
plt.show()

### `Exploratory Visualization of Numerical Attributes`

To explore the distribution and relationships of numerical attributes, I use a combination of histograms, boxplots, and scatter plots. These visualizations help assess data spread, skewness, outliers, and the strength of relationships between numerical features and the target variable. Understanding these patterns is essential for preparing data and selecting appropriate regression modeling strategies.

`Prompt provided from Textbook: "Create charts of the types above considering the 'price' attribute of the dataset." (Moscato 183)`

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Read the CSV file into a DataFrame
df = pd.read_csv("House_Prices.csv")

# Set up the figure and axes
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# Histogram
sns.histplot(df['price'], kde=True, ax=axes[0, 0])
axes[0, 0].set_title('Histogram of Price')
axes[0, 0].set_xlabel('Price')

# Boxplot
sns.boxplot(y='price', data=df, ax=axes[0, 1])
axes[0, 1].set_title('Boxplot of Price')
axes[0, 1].set_ylabel('Price')

# Scatter plot with sqft_living
sns.scatterplot(x='sqft_living', y='price', data=df, ax=axes[0, 2])
axes[0, 2].set_title('Price vs. Square Footage of Living Area')
axes[0, 2].set_xlabel('Square Footage of Living Area')
axes[0, 2].set_ylabel('Price')

# Scatter plot with bedrooms
sns.scatterplot(x='bedrooms', y='price', data=df, ax=axes[1, 0])
axes[1, 0].set_title('Price vs. Number of Bedrooms')
axes[1, 0].set_xlabel('Number of Bedrooms')
axes[1, 0].set_ylabel('Price')

# Scatter plot with bathrooms
sns.scatterplot(x='bathrooms', y='price', data=df, ax=axes[1, 1])
axes[1, 1].set_title('Price vs. Number of Bathrooms')
axes[1, 1].set_xlabel('Number of Bathrooms')
axes[1, 1].set_ylabel('Price')

# Scatter plot with grade
sns.scatterplot(x='grade', y='price', data=df, ax=axes[1, 2])
axes[1, 2].set_title('Price vs. Grade')
axes[1, 2].set_xlabel('Grade')
axes[1, 2].set_ylabel('Price')

# Adjust layout
plt.tight_layout()

# Show the plot
plt.show()

#### `Numerical Feature Analysis and Observations`
The histogram and boxplot of price reveal a right-skewed distribution with several high-value outliers, suggesting that price may benefit from transformation or robust modeling techniques. The scatter plots show a strong positive relationship between price and square footage of living area, reinforcing its importance as a key predictor.

The relationship between price and the number of bedrooms appears more dispersed, indicating that bedroom count alone is not a strong predictor without considering total living space. In contrast, bathrooms and grade exhibit clearer upward trends, with grade showing a particularly strong association with higher prices. These observations suggest that quality-related features and overall size play a more significant role in predicting house prices than simple room counts.


## Part 4: Geospatial Analysis for Regression Model

Geospatial analysis examines how location-based features influence the target variable and overall model behavior. In housing data, geographic location often plays a critical role in price variation due to factors such as neighborhood desirability, access to amenities, and regional market conditions.

In this section, I use latitude and longitude to visualize the spatial distribution of houses in the dataset. Mapping property locations provides insight into geographic clustering, density patterns, and potential location-driven effects that may not be fully captured by numerical attributes alone. Understanding these spatial patterns helps inform feature engineering decisions and highlights the importance of location in regression modeling.

`Prompt provided by Textbook: "Create a map to indicate the location of the houses in the dataset. (Moscato 185)`

This geospatial visualization uses the `Folium` library to create an interactive map that displays the geographic distribution of houses in the dataset. Folium enables the creation of web-based maps that support zooming, panning, and interactive exploration, making it well-suited for location-based analysis.

The dataset is first loaded into a DataFrame, where latitude and longitude serve as the geographic coordinates for each house. The map is initialized by centering it at the average latitude and longitude values, ensuring that the initial view is focused on the region where most properties are located.

Each row in the dataset is then iterated over, and a marker (or circular marker) is placed on the map at the corresponding latitude and longitude. These markers visually represent individual houses and collectively illustrate spatial density and clustering patterns across the region.

Finally, the map object is rendered within the notebook, allowing interactive inspection of housing locations. This visualization provides a spatial perspective that complements numerical and statistical analysis.


In [ ]:
import folium
from folium.plugins import MarkerCluster
import pandas as pd

# Read the CSV file into a DataFrame
df = pd.read_csv("House_Prices.csv")

# Create a map centered at the mean latitude and longitude of the dataset
m = folium.Map(location=[df['lat'].mean(), df['long'].mean()], zoom_start=10)

# Create a MarkerCluster layer for the house locations
marker_cluster = MarkerCluster().add_to(m)

# Add markers for each house
for index, row in df.iterrows():
    folium.Marker([row['lat'], row['long']], popup=f"Price: ${row['price']}").add_to(marker_cluster)

# Save the map to an HTML file
m.save("house_locations_map.html")
display(m)

### `Geospatial Insights`

The geospatial visualization reveals clear clustering of houses, indicating that properties are not evenly distributed across the region. Instead, homes tend to concentrate in specific geographic areas, likely corresponding to urban centers, neighborhoods, or regions with higher housing demand.

This spatial clustering suggests that location plays a significant role in housing prices, even when other numerical features such as square footage or grade are similar. The map highlights why latitude and longitude can be valuable predictors in regression modeling or why location-based features (such as zip code or regional aggregates) may improve predictive performance.

Overall, this analysis reinforces the importance of incorporating geographic information into the regression workflow, either directly through coordinates or indirectly through engineered location-based features.


## `Chapter 9 Summary`

Chapter 9 focused on data preparation and training techniques specific to regression models. Using a housing price dataset, I explored how numerical, categorical, and geospatial features relate to a continuous target variable. The chapter emphasized the importance of exploratory analysis, feature understanding, and visualization as foundational steps before fitting and refining regression models.

Through correlation analysis, numerical visualizations, and geospatial mapping, I examined how size, quality, layout, and location influence housing prices. These steps established a clear understanding of the dataset’s structure and prepared the data for effective regression modeling and fine-tuning in the next chapter.

### What Was Learned

- Learned how exploratory data analysis guides regression modeling decisions by revealing feature relationships, distributions, and potential outliers.
- Identified square footage and grade as strong predictors of housing prices, while recognizing that room counts alone provide weaker predictive power.
- Observed right-skewed price distributions, highlighting the potential need for transformations or robust regression techniques.
- Used scatter plots, histograms, and boxplots to evaluate linearity, dispersion, and variability in numerical features.
- Applied geospatial analysis using latitude and longitude to understand spatial clustering and the role of location in housing prices.
- Gained insight into how geographic patterns motivate location-based feature engineering for regression models.
- Established a solid data preparation foundation for regression model training and fine-tuning in Chapter 10.


# Chapter 10:  House Price Dataset - Fine-Tuning Regression Models

## `Chapter 10 Overview`

This chapter focuses on refining regression models to improve predictive performance and interpretability. Building on the data preparation and baseline modeling steps from the previous chapter, the goal here is to fine-tune model configurations, evaluate competing models, and select the most appropriate approach for the given dataset.

By adjusting hyperparameters and systematically comparing model performance, this chapter emphasizes how thoughtful model selection can lead to more accurate and reliable predictions. In addition, the chapter highlights the importance of interpreting regression results to understand how individual features contribute to model outcomes.

### `Chapter Objectives`
By the end of this chapter, I aim to:
- Prepare regression models for fine-tuning by defining appropriate feature sets and model configurations  
- Apply hyperparameter tuning techniques to optimize regression model performance  
- Evaluate and compare multiple regression models using suitable performance metrics  
- Select the best-performing model based on both accuracy and generalization  
- Interpret regression model outputs to understand feature influence and model behavior  


## Section 1: Model Preparation and Hyperparameters Tuning

In this section, I prepare the dataset for regression model fine-tuning by applying feature standardization. Since many regression algorithms are sensitive to the scale of input features, standardizing numerical variables helps ensure that all features contribute equally during model training and hyperparameter optimization.

The target variable, price, is kept separate from the scaling process to preserve its original units. By standardizing the feature set and saving the transformed dataset, I establish a consistent and reproducible foundation for subsequent regression modeling and hyperparameter tuning steps.

`Prompt provided by Textbook: "Create a python script to apply standardization to the dataset. The target variable is called 'price'." (Moscato 190)`

In [ ]:
import pandas as pd
from sklearn.preprocessing import StandardScaler

# Load the dataset
df = pd.read_csv("House_Prices_Modified.csv")

# Separate the target variable "price" from the features
X = df.drop(columns=['price'])  # Features
y = df['price']  # Target variable

# Standardize the features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Combine the standardized features with the target variable
df_scaled = pd.DataFrame(X_scaled, columns=X.columns)
df_scaled['price'] = y  # Add the target variable back

# Save the standardized dataset to a new CSV file
df_scaled.to_csv("House_Prices_Standardized.csv", index=False)

In [ ]:
df = pd.read_csv("House_Prices_Standardized.csv")
df.head()

### `Training and Testing Sets Creation`

After standardizing the feature set, the next step is to divide the dataset into training and testing subsets. This split allows the regression model to be trained on one portion of the data while reserving a separate subset for unbiased performance evaluation.

In this workflow, 70% of the data is allocated for training and 30% for testing. Using a fixed random state ensures reproducibility, allowing model results and evaluations to remain consistent across multiple runs.


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

# Load the standardized dataset
df = pd.read_csv("House_Prices_Standardized.csv")

# Separate the features (X) and the target variable (y)
X = df.drop(columns=['price'])  # Features
y = df['price']  # Target variable

# Split the dataset into training and testing sets (70% train, 30% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Optional: Save the training and testing sets to CSV files
train_df = pd.concat([X_train, y_train], axis=1)
test_df = pd.concat([X_test, y_test], axis=1)
train_df.to_csv("train_data.csv", index=False)
test_df.to_csv("test_data.csv", index=False)

### `Regression Algorithms for Model Selection`



`Prompt provided by Textbook: "Considering the training and testing variables created previously, create a Python script to test the first 3 algorithms returned in the list above. Present all metrics for evaluating the algorithms. Add a summary at the end with the results of all algorithms. (Moscato 193)`

The 3 algoirthms that will be used: `Linear Regression`, `Ridge Regression` and `Lasso Regression`

* `Linear Regression`: A linear approach to modeling the relationship between a dependent variable and one or more independent variables by fitting a linear equation to observed data.

* `Ridge Regression`: A regularization technique that adds a penalty term to the linear regression objective function to prevent overfitting by shrinking the coefficient estimates towards zero.

* `Lasso Regression`: Another regularization technique similar to Ridge Regression, but it uses the L1 norm penalty term which can lead to sparse coefficient estimates by causing some coefficients to be exactly zero.



In [ ]:
127630.43import pandas as pd
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Load the training and testing datasets
train_data = pd.read_csv("train_data.csv")
test_data = pd.read_csv("test_data.csv")

# Separate features (X) and target variable (y) for training and testing sets
X_train = train_data.drop(columns=['price'])
y_train = train_data['price']
X_test = test_data.drop(columns=['price'])
y_test = test_data['price']

# Initialize regression models
models = {
    "Linear Regression": LinearRegression(),
    "Ridge Regression": Ridge(),
    "Lasso Regression": Lasso()
}

# Train and evaluate each model
results = {}
for name, model in models.items():
    # Train the model
    model.fit(X_train, y_train)

    # Make predictions
    y_pred = model.predict(X_test)

    # Calculate evaluation metrics
    mae = mean_absolute_error(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)

    # Store results
    results[name] = {
        "MAE": mae,
        "MSE": mse,
        "R-squared": r2
    }

# Print results
print("Algorithm\t\tMAE\t\tMSE\t\tR-squared")
print("="*50)
for name, metrics in results.items():
    print(f"{name}\t\t{metrics['MAE']:.2f}\t\t{metrics['MSE']:.2f}\t\t{metrics['R-squared']:.2f}")

# Summary
print("="*50)
print("Summary:")
for name, metrics in results.items():
    print(f"{name}:")
    print(f"\tMean Absolute Error: {metrics['MAE']:.2f}")
    print(f"\tMean Squared Error: {metrics['MSE']:.2f}")
    print(f"\tR-squared: {metrics['R-squared']:.2f}")
    print("-"*30)

### `Regression Model Performance Interpretation`

The performance of three regression algorithms—Linear Regression, Ridge Regression, and Lasso Regression—was evaluated using Mean Absolute Error (MAE), Mean Squared Error (MSE), and R-squared on the standardized test dataset.

* Mean Absolute Error (MAE): MAE represents the average absolute difference between the predicted house prices and the actual house prices in dollars. Lower MAE values indicate better performance, as it means the model's predictions are closer to the actual prices.

* Mean Squared Error (MSE): MSE measures the average squared difference between the predicted house prices and the actual house prices. Similar to MAE, lower MSE values indicate better performance.

* R-squared ($R^2$): R-squared represents the porportion of the variance in the dependent variable (y; house prices) that is predictable from the independent variables (X; features). Higher R-squared values indicate a better fit of the model to the data with 1.0 being a perfect fit.


Linear Regression provides a strong baseline, achieving an MAE of approximately `127630.43`, an MSE of `43413626657.00`, and an R-squared value of `0.70`. These results indicate that the model explains `70%` of the variance in house prices.

Ridge Regression demonstrates a comparable performance, with an MAE of `127628.16`, an MSE of `43413792577.37`, and an R-squared of `0.70`.

Lasso Regression shows an MAE of `127630.56`, an MSE of `43413760727.44`, and an R-squared of `0.70`.

Overall, based on these metrics, we can see that there is no significant difference in performance between the three regression algorithms tested. They all have similar MAE, MSE, and R-squared values. In practice, they perform similarly in predicting house prices for this dataset.


## Section 2: Model Evaluation and Selection - Cross-Validation

`Prompt provided by Textbook: "Write a python script to perform the cross-validation with these three algorithms. Show at the end a summary with the results for each algorithm and the standard deviation. (Moscato 198)`

In [ ]:
from sklearn.model_selection import cross_validate
from sklearn.linear_model import LinearRegression, Ridge, Lasso
import numpy as np

# Define the regression algorithms
linear_regression = LinearRegression()
ridge_regression = Ridge()
lasso_regression = Lasso()

# Define the evaluation metrics
scoring = ['neg_mean_absolute_error', 'neg_mean_squared_error', 'r2']

# Perform cross-validation for Linear Regression
linear_regression_cv_results = cross_validate(linear_regression, X_train, y_train, cv=5, scoring=scoring)

# Perform cross-validation for Ridge Regression
ridge_regression_cv_results = cross_validate(ridge_regression, X_train, y_train, cv=5, scoring=scoring)

# Perform cross-validation for Lasso Regression
lasso_regression_cv_results = cross_validate(lasso_regression, X_train, y_train, cv=5, scoring=scoring)

# Calculate mean and standard deviation for each metric
linear_regression_scores = linear_regression_cv_results['test_neg_mean_absolute_error'], linear_regression_cv_results['test_neg_mean_squared_error'], linear_regression_cv_results['test_r2']
linear_regression_mean = [np.mean(score) for score in linear_regression_scores]
linear_regression_std = [np.std(score) for score in linear_regression_scores]

ridge_regression_scores = ridge_regression_cv_results['test_neg_mean_absolute_error'], ridge_regression_cv_results['test_neg_mean_squared_error'], ridge_regression_cv_results['test_r2']
ridge_regression_mean = [np.mean(score) for score in ridge_regression_scores]
ridge_regression_std = [np.std(score) for score in ridge_regression_scores]

lasso_regression_scores = lasso_regression_cv_results['test_neg_mean_absolute_error'], lasso_regression_cv_results['test_neg_mean_squared_error'], lasso_regression_cv_results['test_r2']
lasso_regression_mean = [np.mean(score) for score in lasso_regression_scores]
lasso_regression_std = [np.std(score) for score in lasso_regression_scores]

# Print the results
print("Results for Linear Regression:")
for metric, mean, std in zip(scoring, linear_regression_mean, linear_regression_std):
    print(f"{metric}: {mean:.2f} (+/- {std:.2f})")
print()

print("Results for Ridge Regression:")
for metric, mean, std in zip(scoring, ridge_regression_mean, ridge_regression_std):
    print(f"{metric}: {mean:.2f} (+/- {std:.2f})")
print()

print("Results for Lasso Regression:")
for metric, mean, std in zip(scoring, lasso_regression_mean, lasso_regression_std):
    print(f"{metric}: {mean:.2f} (+/- {std:.2f})")
print()

### `Results Interpretation`

Results for Linear Regression:
neg_mean_absolute_error: -125059.42 (+/- 514.95)
neg_mean_squared_error: -39573030421.95 (+/- 4785482419.71)
r2: 0.70 (+/- 0.01)

Results for Ridge Regression:
neg_mean_absolute_error: -125056.23 (+/- 514.92)
neg_mean_squared_error: -39572986225.65 (+/- 4785743850.16)
r2: 0.70 (+/- 0.01)

Results for Lasso Regression:
neg_mean_absolute_error: -125059.37 (+/- 514.96)
neg_mean_squared_error: -39573027640.68 (+/- 4785397865.75)
r2: 0.70 (+/- 0.01)

### `Linear Regresion is the selection for the final project.`



## Section 3: Linear Regression Algorithm to Train Model

In [ ]:
from sklearn.linear_model import LinearRegression

# Assuming X_train and y_train are your training features and target variable respectively
# Fit the Linear Regression model
model = LinearRegression()
model.fit(X_train, y_train)

# Get the coefficients
coefficients = model.coef_
print("Coefficients:", coefficients)

# Get the intercept
intercept = model.intercept_
print("Intercept:", intercept)

### Linear Regression Parameters Explanation



### Features Importance of Model

`Prompt provided by Textbook: I want to use a model with the best parameters above and visualize a chart that shows the features importance. (Moscato 203)`

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Coefficients obtained from your Linear Regression model
coefficients = [-32260.38714871, 32989.72340545, 83105.99832004, 4029.30799642,
                3581.64453266, 47545.05685123, 42489.39358804, 13531.76465891,
                112070.67331801, 77598.09941777, 27271.44658501, -80298.9876787,
                -30088.86140082, 82145.85854711, -27382.63081926, 10441.39685728,
                -8828.69733048]
feature_names = ['feat1', 'feat2', 'feat3', 'feat4', 'feat5', 'feat6', 'feat7',
                 'feat8', 'feat9', 'feat10', 'feat11', 'feat12', 'feat13', 'feat14',
                 'feat15', 'feat16', 'feat17']

# Plotting
plt.figure(figsize=(10, 6))
plt.barh(feature_names, np.abs(coefficients), color='skyblue')
plt.xlabel('Coefficient Magnitude')
plt.title('Feature Importance in Linear Regression Model')
plt.gca().invert_yaxis()  # Invert y-axis to have the most important feature at the top
plt.show()

## Part 4: Training Model with Standardized Variables

`Prompt provided by Textbook: "Create a script that trains the model on the entire database using the best parameters and save the model to disk. After that, create another script that loads the model, receives the data provided by the user as input and makes the prediction. The full dataset is the dataframe 'House_Prices_Standardized.csv' and it contains the target variable 'price.' (Mostcato 204)`

In [ ]:
import pandas as pd
from sklearn.linear_model import LinearRegression
import joblib

# Load the standardized dataset
df = pd.read_csv("House_Prices_Standardized.csv")

# Separate features (X) and target variable (y)
X = df.drop(columns=['price'])  # Features
y = df['price']  # Target variable

# Train the model with the best parameters
model = LinearRegression()
model.fit(X, y)

# Save the trained model to disk
joblib.dump(model, 'linear_regression_model.pkl')

print("Model trained and saved successfully!")

## Part 5: Final Regression Model

In [ ]:
import random

random.seed(42)
reproducible_numbers = [random.random() for _ in range(17)]
print("\nReproducible random floats (seed=42):")
print(reproducible_numbers)

In [ ]:
import joblib

# Load the trained model from disk
model = joblib.load('linear_regression_model.pkl')

# Get input data from the user
input_data = []
for i in range(1, 18):
    value = float(input(f"Enter value for feature {i}: "))
    input_data.append(value)

# Make predictions
predicted_price = model.predict([input_data])

print(f"Predicted price: ${predicted_price[0]:,.2f}")

## `Chapter 10 Summary`

In this chapter, I focused on fine-tuning and evaluating regression models for house price prediction. Building on the data preparation and feature analysis performed in Chapter 9, the goal was to improve model performance, compare multiple regression approaches, and select the most appropriate model for the dataset.

The chapter began with standardizing the feature set to ensure consistent scaling across variables, followed by splitting the data into training and testing subsets to support unbiased model evaluation. I then trained and evaluated multiple regression algorithms—Linear Regression, Ridge Regression, and Lasso Regression—using quantitative performance metrics to assess predictive accuracy and model stability.

Through systematic evaluation and comparison, this chapter demonstrates how fine-tuning, regularization, and metric-based evaluation contribute to selecting reliable and interpretable regression models for real-world data analysis tasks.

### `What Was Learned`

- How feature standardization improves regression model training, particularly for regularized models such as Ridge and Lasso regression  
- The importance of separating training and testing data to ensure fair and reproducible model evaluation  
- How to evaluate regression models using Mean Absolute Error (MAE), Mean Squared Error (MSE), and R-squared, and what each metric reveals about model performance  
- The practical differences between Linear Regression, Ridge Regression, and Lasso Regression, including how regularization affects model stability and interpretability  
- How to compare multiple regression models objectively and select a best-performing model based on both accuracy and generalization  
- Why model interpretability is an essential consideration alongside predictive performance in regression analysis  
